In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
# 解决中文显示问题
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows 黑体
plt.rcParams['axes.unicode_minus'] = False   # 解决负号显示问题

input_file = r"D:\2026夏季学期\传播数据分析\数据文件\百度指数数据.csv"

In [ ]:
# 第1步：读取CSV文件
df = pd.read_csv(input_file, encoding='utf-8')

# 核心列适配
df['日期'] = pd.to_datetime(df['日期'])  # 将日期列转换为datetime格式
df['全部'] = pd.to_numeric(df['全部'], errors='coerce')  # 将热度列转换为数值类型，无法转换的设置为NaN

# 清理无效数据
df = df.dropna(subset=['全部'])  # 删除热度列为NaN的行
results = []

# 获取所有不重复的热词名称
keywords = df['关键词'].unique()
print(f"找到{len(keywords)}个热词：{list(keywords)}")

找到74个热词：['厉害了我的哥', '友谊的小船', '吃瓜群众', '洪荒之力', '老司机', '葛优躺', '蓝瘦香菇', '辣眼睛', '你有freestyle吗', '你的良心不会痛吗', '尬聊', '怼', '扎心了老铁', '打call', '油腻', '皮皮虾我们走', '还有这种操作', 'c位', 'skr', '佛系', '土味情话', '官宣', '杠精', '燃烧我的卡路里', '确认过眼神', '锦鲤', '996', '不忘初心', '好嗨哟', '断舍离', '柠檬精', '硬核', '道路千万条安全第一条', '雨女无瓜', '不约而同', '云监工', '光盘行动', '夺冠', '奥利给', '好家伙', '秋天的第一杯奶茶', '逆行者', '集美', 'yyds', '元宇宙', '双减', '强国有我', '破防', '绝绝子', '觉醒年代', '躺平', '一起向未来', '中国式现代化', '俄乌冲突', '党的二十大', '全过程人民民主', '我的眼睛就是尺', '数字经济', '电子榨菜', '多巴胺穿搭', '显眼包', '村ba', '特种兵式旅游', 'city不city', '主理人', '争气', '人工智能', '新质生产力', '浓人淡人', 'deepseek', '情绪价值', '来财', '浪浪山小妖怪', '爱你老己']


In [4]:
# 第2步：对每个热词分别计算生命周期
for keyword in keywords:
    print(f"\n正在处理热词：{keyword}")
    # 筛选出当前热词的数据，按日期排序
    keyword_data = df[df['关键词'] == keyword].copy()
    keyword_data = keyword_data.sort_values('日期') 

    # 剔除热度值为0的行(0值代表无热度)
    keyword_data = keyword_data[keyword_data['全部'] > 0].copy()

    if len(keyword_data) == 0:
        print(f"热词 '{keyword}' 没有有效数据，跳过。")
        continue

    # 计算总热度
    total_hot = keyword_data['全部'].sum()
    # 计算累积百分比
    keyword_data['cum_pct'] = keyword_data['全部'].cumsum() / total_hot * 100

    # 找起点：累积百分比第一次超过5%的日期
    start_candidates = keyword_data[keyword_data['cum_pct'] >= 5]
    if len(start_candidates) > 0:
        start_date = start_candidates.iloc[0]['日期']
    else:
        start_date = None

    # 找终点：累积百分比第一次超过95%的日期
    end_candidates = keyword_data[keyword_data['cum_pct'] >= 95]
    if len(end_candidates) > 0:
        end_date = end_candidates.iloc[0]['日期']
    else:
        end_date = None

    # 计算生命周期天数
    if start_date and end_date:
        lifecycle_days = (end_date - start_date).days
    else:
        lifecycle_days = None

    # 保存结果
    results.append({
        'keyword': keyword,
        'start_date': start_date.strftime('%Y-%m-%d') if start_date else '',
        'end_date': end_date.strftime('%Y-%m-%d') if end_date else '',
        'lifecycle_days': lifecycle_days
    })

    print(f"起点：{start_date.date() if start_date else '无'}, 终点：{end_date.date() if end_date else '无'}, 生命周期：{lifecycle_days}天")


正在处理热词：厉害了我的哥
起点：2016-09-13, 终点：2017-10-24, 生命周期：406天

正在处理热词：友谊的小船
起点：2016-04-13, 终点：2017-10-30, 生命周期：565天

正在处理热词：吃瓜群众
起点：2015-10-09, 终点：2017-10-28, 生命周期：750天

正在处理热词：洪荒之力
起点：2015-09-12, 终点：2017-07-19, 生命周期：676天

正在处理热词：老司机
起点：2015-08-11, 终点：2017-11-16, 生命周期：828天

正在处理热词：葛优躺
起点：2016-07-14, 终点：2017-10-07, 生命周期：450天

正在处理热词：蓝瘦香菇
起点：2016-10-11, 终点：2017-06-04, 生命周期：236天

正在处理热词：辣眼睛
起点：2016-01-05, 终点：2017-10-27, 生命周期：661天

正在处理热词：你有freestyle吗
起点：2017-07-01, 终点：2018-10-07, 生命周期：463天

正在处理热词：你的良心不会痛吗
起点：2017-02-21, 终点：2018-10-04, 生命周期：590天

正在处理热词：尬聊
起点：2017-03-16, 终点：2018-11-05, 生命周期：599天

正在处理热词：怼
起点：2016-04-14, 终点：2018-08-24, 生命周期：862天

正在处理热词：扎心了老铁
起点：2017-02-06, 终点：2018-07-16, 生命周期：525天

正在处理热词：打call
起点：2017-06-26, 终点：2018-07-20, 生命周期：389天

正在处理热词：油腻
起点：2016-05-11, 终点：2018-11-19, 生命周期：922天

正在处理热词：皮皮虾我们走
起点：2017-02-09, 终点：2018-06-11, 生命周期：487天

正在处理热词：还有这种操作
起点：2017-06-01, 终点：2020-03-06, 生命周期：1009天

正在处理热词：c位
起点：2017-07-04, 终点：2019-10-08, 生命周期：826天

正在处理热词：skr
起点：2018-07-15, 终点：2019-0

In [ ]:
# 第3步：生成热度趋势曲线图，标出起点和终点
plt.figure(figsize=(14, 5))
plt.plot(keyword_data['日期'], keyword_data['全部'], label='搜索热度', color='blue', linewidth=1.5)

# 在图上标出起点（绿色竖线）
if start_date:
        plt.axvline(x=start_date, color='green', linestyle='--', label=f'起点 (5%): {start_date.date()}')

# 在图上标出终点（红色竖线）
if end_date:
    plt.axvline(x=end_date, color='red', linestyle='--', label=f'终点 (95%): {end_date.date()}')

plt.title(f'{keyword} 搜索热度趋势（生命周期：{lifecycle_days}天）', fontsize=12)
plt.xlabel('日期')
plt.ylabel('百度指数')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# 保存图像（和CSV文件同目录）
output_dir = os.path.dirname(input_file)
img_path = os.path.join(output_dir, f"{keyword}_lifecycle.png")
plt.savefig(img_path, dpi=150)
plt.close()

print(f"已保存生命周期图像：{img_path}")

IndentationError: unexpected indent (418935611.py, line 2)

In [6]:
# 第4步：将所有热词的生命周期结果保存到CSV文件
results_df = pd.DataFrame(results)
output_csv_path = os.path.join(os.path.dirname(input_file), "lifecycle_start-end.csv")
results_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

print(f"已保存生命周期起点和终点数据：{output_csv_path}")
print(f"共成功处理了{len(results)}个热词。")

NameError: name 'output_csv_path' is not defined